# Isothermal 0D Master Equation Simulation
### Import libraries

In [ ]:
import os
import sys
import pickle
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.notebook import tqdm

Visualization libraries

In [ ]:
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
# %matplotlib qt
COLORS = mpl.rcParams['axes.prop_cycle'].by_key()['color']

Global paths

In [ ]:
# WORKSPACE
WORKSPACE = os.environ['WORKSPACE_PATH']
# SurQCT
SurQCT = WORKSPACE + '/ML/PyCOMET/pycomet/examples/SurQCT/'
sys.path.append(os.path.abspath(SurQCT))
# Current Working Directory
CWD = os.getcwd()

In [ ]:
# Output directory
out_dir = CWD + '/../Output/MasterEquation/'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

### Initializing Run

In [ ]:
SystName       = 'O3_UMN'
ProcessesCode  = '0_1_0_0'
TTranVec       = [1500, 5000, 10000, 20000]
RunCase        = '/KInel/case.01/'

In [ ]:
out_dir        = out_dir + RunCase + ProcessesCode
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

Loading chemical system

In [ ]:
from surqct.src.ChemicalSystems import Systems
System = getattr(Systems, SystName)()
# Update path to SurQCT folder
System.SurQCTMEPath = System.SurQCTMEPath + RunCase

### Reading Master Equation Solution Files

In [ ]:
QCT    = {}
SurQCT = {}

for iTemp in tqdm(range(len(TTranVec))):

    Temp = TTranVec[iTemp]
    
    QCT[Temp] = {}
    SurQCT[Temp] = {}
    
    # QCT
    print(f'\nReading    QCT 0D ME solution at temperature T = {Temp} K')
    PathToRunFldr               = System.QCTMEPath + f'/T{str(Temp)}K/{ProcessesCode}/output_box/'
    QCT[Temp]['box']            = pd.read_csv(PathToRunFldr + 'box.dat', delimiter='  ', skiprows=1, engine='python')
    QCT[Temp]['box'].columns    = System.BoxHeader
    QCT[Temp]['box'].NSteps     = len(QCT[Temp]['box'])
    QCT[Temp]['pop']            = System.read_populations(PathToRunFldr, QCT[Temp]['box'].NSteps)
    
    # SurQCT
    print(f'Reading SurQCT 0D ME solution at temperature T = {Temp} K')
    PathToRunFldr               = System.SurQCTMEPath + f'/T{str(Temp)}K/{ProcessesCode}/output_box/'
    SurQCT[Temp]['box']         = pd.read_csv(PathToRunFldr + 'box.dat', delimiter='  ', skiprows=1, engine='python')
    SurQCT[Temp]['box'].columns = System.BoxHeader
    SurQCT[Temp]['box'].NSteps  = len(SurQCT[Temp]['box'])
    SurQCT[Temp]['pop']         = System.read_populations(PathToRunFldr, QCT[Temp]['box'].NSteps)


### Computing
Energies

In [ ]:
for iTemp in tqdm(range(len(TTranVec))):
    
    Temp = TTranVec[iTemp]
    
    QCT[Temp]['E'] = System.compute_energies(Ni=QCT[Temp]['pop'])
    SurQCT[Temp]['E'] = System.compute_energies(Ni=SurQCT[Temp]['pop'])


### Plotting
Energies

In [ ]:
def plot_energies(
    Path,
    System,
    QCT,
    SurQCT,
    TTranVec,
    xlim=[1.e-10, 1.e-4]
):
    iTemp = 0
    fig = plt.figure()
    plt.plot([], [],'--k', label='$E_{Rot}$')
    plt.plot([], [], '-k', label='$E_{Vib}$')
    plt.plot([], [], label='QCT', c=COLORS[0])
    plt.plot([], [], label='SurQCT', c=COLORS[1])

    for iTemp in range(len(TTranVec)):

        Temp = TTranVec[iTemp]
        
        for iMol, Mol in enumerate(System.Molecule):
            # QCT
            t = QCT[Temp]['box'].time
            plt.plot(t, QCT[Temp]['E']['ERot'][Mol.Name], '--', c=COLORS[0])
            plt.plot(t, QCT[Temp]['E']['EVib'][Mol.Name],  '-', c=COLORS[0])

            # SurQCT
            t = SurQCT[Temp]['box'].time
            plt.plot(t, SurQCT[Temp]['E']['ERot'][Mol.Name], '--', c=COLORS[1])
            plt.plot(t, SurQCT[Temp]['E']['EVib'][Mol.Name],  '-', c=COLORS[1])
            
            # Temperature annotation
            xy = (1, max(
                QCT[Temp]['E']['ERot'][Mol.Name][-1],
                SurQCT[Temp]['E']['ERot'][Mol.Name][-1]
            ))
            plt.annotate(
                'T = %d K' % Temp,
                xy=xy,
                xytext=(8, 0),
                xycoords=('axes fraction', 'data'),
                textcoords='offset points'
            )

        plt.legend()
        plt.xscale('log')
        plt.xlabel('$t$ [s]')
        plt.ylabel('$E$ [eV]')
        plt.xlim(xlim)

    plt.savefig(Path + f'/EVib_ERot.pdf', bbox_inches="tight")


In [ ]:
plot_energies(out_dir, System, QCT, SurQCT, TTranVec)

Populations

In [ ]:
samples   = [1e-7, 5e-7, 1e-6, 5e-6]
TimeShots = {
    1500:  samples,
    5000:  samples,
    10000: samples,
    20000: samples
}

In [ ]:
# Population plot times
def find_nearest(array, value):
    i = 0
    iStep=[]
    for ivalue in value:
        array = np.asarray(array)
        idx = (np.abs(array - ivalue)).argmin()
        iStep.append(idx)
        i = i+1
    return iStep

def plot_populations(
    Path,
    TempVec,
    TimeShots,
    QCT,
    SurQCT,
    subplots_shape=(1,1),
    figsize_factor=1.5
):
    for iTemp in range(len(TempVec)):
        Temp = TempVec[iTemp]
        QCTStepsVec    = find_nearest(   QCT[Temp]['box'].time, TimeShots[int(Temp)])                                                                                                                                                                               
        SurQCTStepsVec = find_nearest(SurQCT[Temp]['box'].time, TimeShots[int(Temp)])

        fig, ax = plt.subplots(*subplots_shape, sharex=True, figsize=np.array([6.4,4.8])*figsize_factor)
        ax = np.array(ax).reshape(-1)

        for iStep in range(len(TimeShots[int(Temp)])):
            for iMol, Mol in enumerate(System.Molecule):
                cp = ax[iStep].scatter(Mol.EInt - Mol.DissE,    QCT[Temp]['pop'][Mol.Name][   QCTStepsVec[iStep],:]/Mol.g, c='k', s=2, label='QCT')                                                                                                                                
                cp = ax[iStep].scatter(Mol.EInt - Mol.DissE, SurQCT[Temp]['pop'][Mol.Name][SurQCTStepsVec[iStep],:]/Mol.g, c='r', s=2, label='SurQCT', alpha=0.25)
            ax[iStep].set_yscale('log')
            ax[iStep].set_title('$t$ = %.1e s' % TimeShots[int(Temp)][iStep])
            
        fig.add_subplot(111, frame_on=False)
        plt.tick_params(labelcolor='none', which='both', top=False, bottom=False, left=False, right=False)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel('$\epsilon_i$ [eV]', labelpad=30)
        plt.ylabel('$N_i/g_i$ [m$^{-3}$]', labelpad=40)
        
        lines, labels  = ax[0].get_legend_handles_labels()
        plt.legend(lines, labels, frameon=False, markerscale=2.0, ncol=1, bbox_to_anchor=(1., 1.), handletextpad=0.1)
        plt.savefig(Path + f'/Pop_T{Temp}K.pdf', bbox_inches="tight")
        

In [ ]:
plot_populations(out_dir, TTranVec, TimeShots, QCT, SurQCT, subplots_shape=(2,2))

## 